# NatyaVeda — 04 Model Analysis
Inspect trained DanceFormer — attention maps, embeddings, confusion matrix.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.models.danceformer import DanceFormer

DANCE_CLASSES = [
    'bharatanatyam','kathak','odissi','kuchipudi',
    'manipuri','mohiniyattam','sattriya','kathakali'
]

# Load trained model
ckpt = torch.load('../weights/danceformer_best.pt', map_location='cpu')
model = DanceFormer.from_config(ckpt['config'])
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f'Model loaded — {model.count_parameters():,} parameters')
print(f'Best val F1: {ckpt["val_f1"]:.4f}')

In [ ]:
# Load evaluation report and plot confusion matrix
import json
with open('../reports/evaluation_report.json') as f:
    report = json.load(f)

cm = np.array(report['confusion_matrix'])
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f',
            xticklabels=[d[:5] for d in DANCE_CLASSES],
            yticklabels=[d[:5] for d in DANCE_CLASSES],
            cmap='Blues')
plt.title('DanceFormer — Confusion Matrix (Normalized)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout(); plt.show()

print(f"Overall Accuracy: {report['accuracy']:.4f}")
print(f"F1 (weighted):    {report['f1_weighted']:.4f}")

In [ ]:
# Per-class F1 bar chart
per_class = report['per_class']
f1_scores = [per_class.get(d, {}).get('f1-score', 0) for d in DANCE_CLASSES]

colors = plt.cm.RdYlGn([s for s in f1_scores])
plt.figure(figsize=(10, 4))
bars = plt.bar(DANCE_CLASSES, f1_scores, color=colors)
plt.ylim(0, 1.05)
plt.axhline(0.9, color='gray', linestyle='--', alpha=0.5, label='0.90 target')
for bar, score in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.2f}', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.ylabel('F1 Score'); plt.title('Per-Class F1 Scores')
plt.legend(); plt.tight_layout(); plt.show()